In [14]:
%%writefile C:/Users/tseet/OneDrive/Desktop/motor_insurance_analytics/src/insurance_analysis.py
import pandas as pd


# -------------------- SHARED HELPERS --------------------

def _get_table(data, table_name):
    """Return a copy of a required dataset."""
    if table_name not in data:
        raise KeyError(f"Dataset not found: {table_name}")

    df = data[table_name]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{table_name} must be a DataFrame")

    return df.copy()


def _numeric(df, column):
    """Convert a column to numeric values."""
    if column not in df.columns:
        raise ValueError(f"Required column not found: {column}")

    return pd.to_numeric(df[column], errors="coerce")


def _safe_divide(numerator, denominator):
    """Prevent division-by-zero errors."""
    if pd.isna(denominator) or denominator == 0:
        return 0

    return numerator / denominator


def _status(df, column):
    """Normalize status text for reliable comparisons."""
    return (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
        .fillna("")
    )


def _count_status_records(df, id_column, status_column, pattern):
    """Count unique records matching a status pattern."""
    mask = _status(df, status_column).str.contains(
        pattern, regex=True, na=False
    )

    if id_column in df.columns:
        return int(df.loc[mask, id_column].nunique())

    return int(mask.sum())


def _group_claims(data, group_column):
    """Summarize claim count and amounts by a category."""
    claims = _get_table(data, "claims")
    claims["claim_amount"] = _numeric(claims, "claim_amount")

    return (
        claims.groupby(group_column, dropna=False)
        .agg(
            claim_count=("claim_id", "nunique"),
            total_claim_amount=("claim_amount", "sum"),
            average_claim_amount=("claim_amount", "mean"),
        )
        .sort_values("total_claim_amount", ascending=False)
        .reset_index()
    )


# -------------------- PORTFOLIO ANALYSIS --------------------

def calculate_total_customers(data):
    customers = _get_table(data, "customers")
    return int(customers["customer_id"].nunique())


def calculate_total_policies(data):
    policies = _get_table(data, "policies")
    return int(policies["policy_id"].nunique())


def calculate_active_policies(data):
    policies = _get_table(data, "policies")
    return _count_status_records(
        policies, "policy_id", "policy_status", r"^active$"
    )


def calculate_expired_policies(data):
    policies = _get_table(data, "policies")
    return _count_status_records(
        policies, "policy_id", "policy_status", r"expired"
    )


def calculate_cancelled_policies(data):
    policies = _get_table(data, "policies")
    return _count_status_records(
        policies, "policy_id", "policy_status", r"cancelled|canceled"
    )


def calculate_renewed_policies(data):
    policies = _get_table(data, "policies")
    return _count_status_records(
        policies, "policy_id", "policy_status", r"renew"
    )


# -------------------- PREMIUM ANALYSIS --------------------

def calculate_total_premium(data):
    policies = _get_table(data, "policies")
    return float(_numeric(policies, "premium_amount").sum())


def calculate_average_premium(data):
    policies = _get_table(data, "policies")
    return float(_numeric(policies, "premium_amount").mean())


def calculate_average_coverage(data):
    policies = _get_table(data, "policies")
    return float(_numeric(policies, "coverage_amount").mean())


def calculate_premium_by_policy_type(data):
    policies = _get_table(data, "policies")
    policies["premium_amount"] = _numeric(
        policies, "premium_amount"
    )

    return (
        policies.groupby("policy_type", dropna=False)["premium_amount"]
        .sum()
        .sort_values(ascending=False)
    )


# -------------------- CLAIM ANALYSIS --------------------

def calculate_total_claims(data):
    claims = _get_table(data, "claims")
    return int(claims["claim_id"].nunique())


def calculate_approved_claims(data):
    claims = _get_table(data, "claims")
    return _count_status_records(
        claims, "claim_id", "claim_status", r"approv"
    )


def calculate_rejected_claims(data):
    claims = _get_table(data, "claims")
    return _count_status_records(
        claims, "claim_id", "claim_status", r"reject|denied|declin"
    )


def calculate_pending_claims(data):
    claims = _get_table(data, "claims")
    return _count_status_records(
        claims,
        "claim_id",
        "claim_status",
        r"pending|under review|in review",
    )


def calculate_total_claim_amount(data):
    claims = _get_table(data, "claims")
    return float(_numeric(claims, "claim_amount").sum())


def calculate_average_claim_amount(data):
    claims = _get_table(data, "claims")
    return float(_numeric(claims, "claim_amount").mean())


def calculate_claim_approval_rate(data):
    """Approved claims / total claims × 100."""
    total = calculate_total_claims(data)
    approved = calculate_approved_claims(data)

    return float(_safe_divide(approved, total) * 100)


def calculate_claim_rejection_rate(data):
    """Rejected claims / total claims × 100."""
    total = calculate_total_claims(data)
    rejected = calculate_rejected_claims(data)

    return float(_safe_divide(rejected, total) * 100)


# -------------------- SETTLEMENT ANALYSIS --------------------

def calculate_average_claim_processing_days(data):
    """Average days from claim registration to approval."""
    claims = _get_table(data, "claims")

    claim_dates = pd.to_datetime(
        claims["claim_date"], errors="coerce"
    )
    approval_dates = pd.to_datetime(
        claims["approval_date"], errors="coerce"
    )

    days = (approval_dates - claim_dates).dt.days
    valid_days = days[days >= 0].dropna()

    return float(valid_days.mean())


def calculate_average_claim_settlement_days(data):
    """Average days from claim registration to settlement."""
    claims = _get_table(data, "claims")

    claim_dates = pd.to_datetime(
        claims["claim_date"], errors="coerce"
    )
    settlement_dates = pd.to_datetime(
        claims["settlement_date"], errors="coerce"
    )

    days = (settlement_dates - claim_dates).dt.days
    valid_days = days[days >= 0].dropna()

    return float(valid_days.mean())


def calculate_total_payment_amount(data):
    payments = _get_table(data, "payments")
    return float(_numeric(payments, "payment_amount").sum())


def calculate_average_payment_amount(data):
    payments = _get_table(data, "payments")
    return float(_numeric(payments, "payment_amount").mean())


# -------------------- RISK AND RATIOS --------------------

def calculate_claim_to_premium_ratio(data):
    """Total claim amount divided by total premium."""
    claim_amount = calculate_total_claim_amount(data)
    premium = calculate_total_premium(data)

    return float(_safe_divide(claim_amount, premium))


def calculate_claim_to_coverage_ratio(data):
    """Total claim amount divided by total policy coverage."""
    claims = calculate_total_claim_amount(data)
    policies = _get_table(data, "policies")
    coverage = float(_numeric(policies, "coverage_amount").sum())

    return float(_safe_divide(claims, coverage))


def calculate_paid_claim_ratio(data):
    """Total payment amount divided by total claim amount."""
    payment_amount = calculate_total_payment_amount(data)
    claim_amount = calculate_total_claim_amount(data)

    return float(_safe_divide(payment_amount, claim_amount))


# -------------------- CUSTOMER ANALYSIS --------------------

def analyze_customer_claims(data):
    """Return customer-level claim counts and claim amounts."""
    customers = _get_table(data, "customers")
    claims = _get_table(data, "claims")

    claims["claim_amount"] = _numeric(claims, "claim_amount")

    summary = (
        claims.groupby("customer_id", dropna=False)
        .agg(
            claim_count=("claim_id", "nunique"),
            total_claim_amount=("claim_amount", "sum"),
            average_claim_amount=("claim_amount", "mean"),
        )
        .reset_index()
    )

    return customers.merge(
        summary, on="customer_id", how="left"
    ).assign(
        claim_count=lambda df: df["claim_count"].fillna(0),
        total_claim_amount=lambda df: df["total_claim_amount"].fillna(0),
    )


def analyze_customer_premium(data):
    """Return customer-level policy and premium summaries."""
    customers = _get_table(data, "customers")
    policies = _get_table(data, "policies")

    policies["premium_amount"] = _numeric(
        policies, "premium_amount"
    )

    summary = (
        policies.groupby("customer_id", dropna=False)
        .agg(
            policy_count=("policy_id", "nunique"),
            total_premium=("premium_amount", "sum"),
            average_premium=("premium_amount", "mean"),
        )
        .reset_index()
    )

    return customers.merge(
        summary, on="customer_id", how="left"
    ).assign(
        policy_count=lambda df: df["policy_count"].fillna(0),
        total_premium=lambda df: df["total_premium"].fillna(0),
    )


# -------------------- VEHICLE ANALYSIS --------------------

def _vehicle_claim_detail(data):
    """Connect claims to vehicle details through policies."""
    claims = _get_table(data, "claims")
    policies = _get_table(data, "policies")
    vehicles = _get_table(data, "vehicles")

    claims["claim_amount"] = _numeric(
        claims, "claim_amount"
    )

    policy_vehicle = policies[
        ["policy_id", "vehicle_id"]
    ].drop_duplicates("policy_id")

    vehicle_columns = [
        "vehicle_id",
        "vehicle_type",
        "vehicle_make",
        "vehicle_model",
        "vehicle_year",
        "fuel_type",
    ]

    claim_detail = claims.merge(
        policy_vehicle, on="policy_id", how="left"
    ).merge(
        vehicles[vehicle_columns].drop_duplicates("vehicle_id"),
        on="vehicle_id",
        how="left",
    )

    return claim_detail


def analyze_vehicle_claims(data):
    """Return claim metrics for each vehicle."""
    vehicles = _get_table(data, "vehicles")
    detail = _vehicle_claim_detail(data)

    summary = (
        detail.groupby("vehicle_id", dropna=False)
        .agg(
            claim_count=("claim_id", "nunique"),
            total_claim_amount=("claim_amount", "sum"),
            average_claim_amount=("claim_amount", "mean"),
        )
        .reset_index()
    )

    return vehicles.merge(
        summary, on="vehicle_id", how="left"
    ).assign(
        claim_count=lambda df: df["claim_count"].fillna(0),
        total_claim_amount=lambda df: df["total_claim_amount"].fillna(0),
    )


def analyze_vehicle_risk_patterns(data):
    """Compare vehicle claim amounts with premium amounts."""
    vehicles = _get_table(data, "vehicles")
    policies = _get_table(data, "policies")

    policies["premium_amount"] = _numeric(
        policies, "premium_amount"
    )

    premium_summary = (
        policies.groupby("vehicle_id", dropna=False)
        .agg(
            policy_count=("policy_id", "nunique"),
            total_premium=("premium_amount", "sum"),
        )
        .reset_index()
    )

    vehicle_summary = analyze_vehicle_claims(data).merge(
        premium_summary, on="vehicle_id", how="left"
    )

    vehicle_summary["total_premium"] = (
        vehicle_summary["total_premium"].fillna(0)
    )

    vehicle_summary["claim_to_premium_ratio"] = (
        vehicle_summary["total_claim_amount"]
        / vehicle_summary["total_premium"].where(
            vehicle_summary["total_premium"] != 0
        )
    )

    current_year = pd.Timestamp.today().year
    vehicle_summary["vehicle_age"] = (
        current_year
        - pd.to_numeric(vehicle_summary["vehicle_year"], errors="coerce")
    )

    return vehicle_summary.sort_values(
        "claim_to_premium_ratio", ascending=False, na_position="last"
    )


# -------------------- BUSINESS ANALYSIS --------------------

def analyze_claim_type(data):
    claims = _get_table(data, "claims")
    return _group_claims(data, "claim_type")


def analyze_claim_status(data):
    claims = _get_table(data, "claims")
    return _group_claims(data, "claim_status")


def analyze_policy_type(data):
    policies = _get_table(data, "policies")
    policies["premium_amount"] = _numeric(
        policies, "premium_amount"
    )

    return (
        policies.groupby("policy_type", dropna=False)
        .agg(
            policy_count=("policy_id", "nunique"),
            total_premium=("premium_amount", "sum"),
            average_premium=("premium_amount", "mean"),
        )
        .sort_values("total_premium", ascending=False)
        .reset_index()
    )


def analyze_region(data):
    """Summarize claims by the customer's state and city."""
    claims = _get_table(data, "claims")
    customers = _get_table(data, "customers")

    claims["claim_amount"] = _numeric(
        claims, "claim_amount"
    )

    detail = claims.merge(
        customers[["customer_id", "state", "city"]].drop_duplicates(
            "customer_id"
        ),
        on="customer_id",
        how="left",
    )

    return (
        detail.groupby(["state", "city"], dropna=False)
        .agg(
            claim_count=("claim_id", "nunique"),
            total_claim_amount=("claim_amount", "sum"),
            average_claim_amount=("claim_amount", "mean"),
        )
        .sort_values("total_claim_amount", ascending=False)
        .reset_index()
    )


def analyze_vehicle_type(data):
    """Summarize vehicle counts and claim behavior by vehicle type."""
    vehicles = _get_table(data, "vehicles")
    detail = _vehicle_claim_detail(data)

    vehicle_counts = (
        vehicles.groupby("vehicle_type", dropna=False)["vehicle_id"]
        .nunique()
        .rename("vehicle_count")
    )

    claim_summary = (
        detail.groupby("vehicle_type", dropna=False)
        .agg(
            claim_count=("claim_id", "nunique"),
            total_claim_amount=("claim_amount", "sum"),
            average_claim_amount=("claim_amount", "mean"),
        )
    )

    return (
        vehicle_counts.to_frame()
        .join(claim_summary, how="outer")
        .fillna(0)
        .sort_values("total_claim_amount", ascending=False)
        .reset_index()
    )


def analyze_damage_severity(data):
    return _group_claims(data, "damage_severity")


# -------------------- KPI SUMMARY --------------------

def calculate_all_kpis(data):
    """Collect the main required KPIs for dashboards and reports."""
    return {
        "Total Customers": calculate_total_customers(data),
        "Total Policies": calculate_total_policies(data),
        "Active Policies": calculate_active_policies(data),
        "Expired Policies": calculate_expired_policies(data),
        "Cancelled Policies": calculate_cancelled_policies(data),
        "Renewed Policies": calculate_renewed_policies(data),
        "Total Premium": calculate_total_premium(data),
        "Average Premium": calculate_average_premium(data),
        "Average Coverage": calculate_average_coverage(data),
        "Total Claims": calculate_total_claims(data),
        "Approved Claims": calculate_approved_claims(data),
        "Rejected Claims": calculate_rejected_claims(data),
        "Pending Claims": calculate_pending_claims(data),
        "Total Claim Amount": calculate_total_claim_amount(data),
        "Average Claim Amount": calculate_average_claim_amount(data),
        "Claim Approval Rate (%)": calculate_claim_approval_rate(data),
        "Claim Rejection Rate (%)": calculate_claim_rejection_rate(data),
        "Average Processing Days": calculate_average_claim_processing_days(data),
        "Average Settlement Days": calculate_average_claim_settlement_days(data),
        "Total Payment Amount": calculate_total_payment_amount(data),
        "Average Payment Amount": calculate_average_payment_amount(data),
        "Claim-to-Premium Ratio": calculate_claim_to_premium_ratio(data),
        "Claim-to-Coverage Ratio": calculate_claim_to_coverage_ratio(data),
        "Paid Claim Ratio": calculate_paid_claim_ratio(data),
    }